# 02. Huấn luyện Mô hình, Đánh giá Chéo & Tinh chỉnh Siêu tham số
**Học phần:** Học máy (Machine Learning) - HUCE  
**Bài toán:** Phân loại nhị phân dự đoán `Machine failure`  
**Mục tiêu:** Xây dựng Scikit-learn Pipeline chuẩn, thẩm định chéo Stratified K-Fold (tránh rò rỉ dữ liệu), thử nghiệm $\ge 3$ dòng thuật toán, tinh chỉnh siêu tham số và xuất file `best_model.joblib`.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Các thuật toán thử nghiệm (ít nhất 3 dòng)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

## 1. Nạp dữ liệu và Tách tập Train / Test độc lập (Chống Data Leakage)
Tập Test được cô lập hoàn toàn (stratified split 80/20) và chỉ được dùng ở bước đánh giá cuối cùng.

In [ ]:
data_path = os.path.join("..", "data", "ai4i+2020+predictive+maintenance+dataset", "ai4i2020.csv")
df = pd.read_csv(data_path)

# Định nghĩa X và y (Loại bỏ ID và các cờ lỗi nguyên nhân)
feature_cols = ["Type", "Air temperature [K]", "Process temperature [K]", 
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
X = df[feature_cols]
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Kích thước X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. Xây dựng Pipeline tiền xử lý với `ColumnTransformer`

In [ ]:
num_features = ["Air temperature [K]", "Process temperature [K]", 
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
cat_features = ["Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_features)
    ]
)

## 3. Thử nghiệm $\ge 3$ dòng mô hình & Đánh giá Stratified K-Fold CV
Thử nghiệm:
1. **Logistic Regression** (Tuyến tính, có `class_weight='balanced'`)
2. **Decision Tree** (Mô hình cây phân nhánh)
3. **Random Forest** (Mô hình tập hợp Ensemble)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Nhóm sẽ hoàn thiện vòng lặp đánh giá chéo đa chỉ số tại đây

## 4. Tinh chỉnh Siêu tham số (GridSearchCV) cho mô hình tốt nhất

In [ ]:
# Khung cấu hình GridSearchCV trên toàn bộ Pipeline
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42))
])

param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 10, 20]
}

grid_search = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f"Best Parameters: {grid_search.best_params_}")

## 5. Đánh giá khách quan trên Test Set & Xuất file mô hình (`.joblib`)

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("=== BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ĐỘC LẬP ===")
print(classification_report(y_test, y_pred))

# Lưu mô hình xuất bản
output_model_path = os.path.join("..", "models", "best_model.joblib")
os.makedirs(os.path.dirname(output_model_path), exist_ok=True)
joblib.dump(best_model, output_model_path)
print(f"Đã xuất xưởng mô hình thành công tại: {output_model_path}")